# Mask2Former Swin-T COCO panoptic — DIMER end-to-end adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mask2former-panoptic-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mask2former-panoptic-pipeline/blob/main/tutorials/mask2former_panoptic_finetune_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fmask2former--swin--tiny--coco--panoptic-ffcc4d?style=flat)](https://huggingface.co/facebook/mask2former-swin-tiny-coco-panoptic) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2FMask2Former-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/Mask2Former) [![arXiv](https://img.shields.io/badge/arXiv-2112.01527-b31b1b.svg)](https://arxiv.org/abs/2112.01527)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** bounded gradient adaptation of the pinned `facebook/mask2former-swin-tiny-coco-panoptic` weights to a caller-supplied panoptic vocabulary (five drawn classes: two stuff, three things) — dataset validation, seeded split, re-headed baseline, fine-tune with the upstream set loss, held-out panoptic quality, new-data inference, artifact export and fresh reload

**This notebook is standalone.** It carries the repository's package (2 modules under `src/mask2former_panoptic_pipeline/`, at revision `9dc0dc7df800`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `df6b1142ff50c3276559d9d78f35f6a579c75a77` (~190 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, shows what the COCO-vocabulary checkpoint does on a drawn scene, draws and validates a 24-image labelled panoptic dataset, splits it 18/6 with a seed, rebuilds the classification head for the five-class vocabulary and measures the held-out baseline (plus a trivial all-`sky` baseline), **runs the bounded fine-tune** (6 epochs, 54 optimizer steps, backbone frozen), re-scores the held-out split with panoptic quality, segments unseen scenes, exports the adapted model as a self-describing directory, reloads it from disk and confirms the reloaded model reproduces the held-out score exactly, and writes every result as JSON with provenance. Nothing is skipped behind a default-off flag, and the path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5, RUN7, FT2). It runs on CPU in a few minutes (the repository's smoke run trained in 185 s on the reference workstation) and faster on a CUDA GPU.

**Bring Your Own Data:** Two BYOD branches are provided and both are optional and off by default. `USE_BYOD_IMAGE` runs your own image through the adapted model with the same validation and inference contract as the drawn scenes. `USE_BYOD_DATASET` takes your own labelled panoptic records (a directory with `dataset.json`, images and instance-id PNGs, read by the carried `load_labelled_dir`) and runs them through the *same* local stages the sample used — validate, split, baseline, fine-tune, evaluate, export, reload — rather than only segmenting with them, because this is an adaptation profile (NOTEBOOK_SPEC 2.0 §25.10). The expected record shape and the ceilings are stated in the Prerequisites and printed by the cell; uploads stay inside this runtime.

Mask2Former treats panoptic, instance and semantic segmentation as one problem: predict a set of (class, mask) pairs. That makes its head cheap to replace — the class predictor is one linear layer over the query embeddings — while the Swin-T backbone, the multi-scale deformable-attention pixel decoder and the masked-attention transformer decoder keep everything they learned on COCO. This notebook does exactly that: `from_pretrained(class_names=..., stuff_names=...)` loads the same digest-verified checkpoint, rebuilds the class head for a five-class drawn vocabulary (`sky` and `ground` as stuff, `disc`, `box` and `triangle` as things) with a seeded random initialisation of exactly the mismatched tensors, and `finetune` runs a bounded plain-PyTorch AdamW loop with the **upstream set loss** (Hungarian matching of queries to instances, class cross-entropy, point-sampled mask BCE and dice, auxiliary decoder losses) on 18 drawn scenes, backbone frozen. **What is trained and what is not:** the transformer decoder, the pixel decoder and the new class head (19.9 M of 47.4 M parameters) train; the Swin-T backbone is frozen and kept in eval mode. Evaluation is panoptic quality on 6 held-out scenes through the same `segment` call as inference, before and after adaptation, beside a trivial all-`sky` baseline. **Training loss is optimisation evidence only** (FT7); the held-out PQ is the task evidence, and it is a sample metric on drawn shapes, not a benchmark. The artifact is the full adapted model (SafeTensors + configs + a manifest with digests and provenance), reloaded from disk by `load_artifact` and re-scored to prove the boundary.

**Learning objectives:** read what the carried package guarantees, verify the pinned base weights, see the COCO checkpoint fail on drawn shapes, draw and validate a labelled panoptic dataset into a dataset manifest, split it with a seed, re-head the model for a new vocabulary and measure a baseline before touching any weight, run a bounded fine-tune with explicit hyperparameters and read its loss history for what it is, evaluate with panoptic quality on held-out scenes and compare against the baselines, segment new scenes, export a self-describing artifact, reload it from disk and prove it reproduces the held-out score, and export machine-readable results with provenance.

**This notebook does not demonstrate:** Adapting the backbone (frozen here; unfreezing it on 18 images is a recipe for forgetting), the upstream training recipe (COCO, 50 epochs, 12,544 sampled points per mask, large-scale jitter), any photographic dataset (the drawn scenes are the whole sample; PQ on them says the plumbing works, not that the model would learn your photographs), hyperparameter search, mixed precision, multi-GPU training, calibration, and any claim about COCO panoptic quality. The pretrained COCO vocabulary is not preserved by the adaptation: the re-headed model knows only the five new classes.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; training and inference are float32 on both. CPU is adequate for the default path: the repository's model card records 185 s for the 54-step fine-tune (about 3.4 s per step at 384×384, batch 2) and 3–6 s per 6-image evaluation in the Windows venv (Intel Core Ultra 9 275HX); a T4 is several times faster. The pinned `torch==2.14.0` install and the 190 MB checkpoint are the large downloads of the run; the exported artifact is another 190 MB on disk.
- **Knowledge:** basic Python, NumPy and PIL; what a panoptic annotation is (an instance map plus a class per instance) and why stuff and things are treated differently; what a train/held-out split protects against; how set-prediction losses match queries to targets; what panoptic quality (PQ = SQ × RQ, IoU > 0.5 matching) measures.
- **Data:** the default sample is drawn in code by the carried `shape_dataset` (24 scenes of 320×240 with a seeded generator: a `sky` region above a jittered horizon, a `ground` band below it and 1–3 things of jittered size, position and colour) with exact instance maps, so nothing is downloaded and no private data is needed; new-data scenes use a different seed. Expected BYOD dataset: a directory with `dataset.json` (`class_names`, `stuff_names`, `records` of `{id, image, mask, instances}`), one image per record decodable by Pillow, and one PNG instance map per record whose pixel values are instance ids in 1..254 with every pixel assigned; ceilings: 2..32 classes, at most 200 records, at most 50 instances per image, at most 20 epochs. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/mask2former-swin-tiny-coco-panoptic` snapshot (~190 MB in total) at revision `df6b1142ff50…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'mask2former-panoptic-pipeline',
    'repository_revision': '9dc0dc7df80002fd67afe76d7f7e6f3802ae68e9',
    'embedded_module': 'src/mask2former_panoptic_pipeline/pipeline.py',
    'embedded_modules': ['src/mask2former_panoptic_pipeline/samples.py', 'src/mask2former_panoptic_pipeline/pipeline.py'],
    'module_sha256': '886084d3e7c028bd0095721704fa32862b71dcca55da9de4acc116419446e9b5',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/mask2former_panoptic_pipeline/` @ `9dc0dc7df800`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/mask2former_panoptic_pipeline/samples.py`

In [ ]:
"""Drawn sample data for the panoptic pipeline: the single inference scene and the labelled shape dataset.

Everything here is deterministic Pillow drawing (no text rendering, no randomness beyond a seeded NumPy
generator), so the samples are reproducible byte-for-byte and carry their own exact references: every
region is known because it was drawn. A **panoptic** reference is an instance map (``int32``, one id per
region, ``0`` nowhere) plus a mapping from instance id to class index; ``stuff`` classes are amorphous
regions (sky, ground) and ``thing`` classes are countable objects (disc, box, triangle).

Nothing in this module imports a model library; it is pure NumPy + Pillow so validation runs before any
model dependency is touched (fleet RTM-001).
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageDraw

# The adaptation vocabulary of the drawn dataset: two stuff classes then three thing classes.
SHAPE_CLASSES: tuple[str, ...] = ("sky", "ground", "disc", "box", "triangle")
SHAPE_STUFF: tuple[str, ...] = ("sky", "ground")
# Dataset ceilings enforced by validate_dataset (and therefore by finetune).
MAX_DATASET_IMAGES = 200
MAX_INSTANCES_PER_IMAGE = 50
# Instance ids are pixel values: 0 means no instance and 255 is the pinned processor's ignore index.
MAX_INSTANCE_ID = 254
MIN_CLASSES = 2
MAX_CLASSES = 32
MAX_EPOCHS = 20
MAX_CLASS_NAME_CHARS = 32


def _shape(draw: ImageDraw.ImageDraw, kind: str, geometry: Any, fill: Any) -> None:
    getattr(draw, kind)(geometry, fill=fill)


def synthetic_scene(
    width: int = 640, height: int = 480
) -> tuple[Image.Image, np.ndarray, list[dict[str, Any]]]:
    """The inference-tutorial scene: a red disc, a blue box, a yellow triangle and a green ground band on
    an off-white sky. Returns ``(image, reference_segmentation, reference_segments)`` where the
    segmentation is an ``int32`` map of region ids (1..5, no void) and each segment records its id, a
    region name, whether it is stuff, and its area fraction. Later shapes overdraw earlier ones, and the
    references are computed after overdrawing so they are exact."""
    image = Image.new("RGB", (width, height), (245, 245, 240))
    ids = Image.new("I", (width, height), 1)  # 1 = sky (everything not covered below)
    d, di = ImageDraw.Draw(image), ImageDraw.Draw(ids)
    regions = [
        (2, "ground", True, "rectangle", [0, 320, width, height], (60, 179, 75)),
        (3, "disc", False, "ellipse", [80, 80, 260, 260], (220, 40, 40)),
        (4, "box", False, "rectangle", [340, 90, 560, 300], (40, 70, 200)),
        (5, "triangle", False, "polygon", [(200, 460), (320, 330), (440, 460)], (250, 200, 30)),
    ]
    for region_id, _name, _stuff, kind, geometry, colour in regions:
        _shape(d, kind, geometry, colour)
        _shape(di, kind, geometry, region_id)
    segmentation = np.array(ids, dtype=np.int32)
    segments = [{"id": 1, "name": "sky", "label": "sky", "is_thing": False}] + [
        {"id": region_id, "name": name, "label": name, "is_thing": not stuff}
        for region_id, name, stuff, *_ in regions
    ]
    for segment in segments:
        segment["area_fraction"] = float((segmentation == segment["id"]).mean())
    return image, segmentation, segments


def shape_dataset(
    n_images: int = 24,
    seed: int = 0,
    width: int = 320,
    height: int = 240,
) -> list[dict[str, Any]]:
    """Draw ``n_images`` labelled panoptic scenes over SHAPE_CLASSES with a seeded generator.

    Each record is ``{"id", "image", "instance_map", "instances"}``: the image, an ``int32`` instance
    map (every pixel belongs to exactly one instance) and ``{instance_id: class_name}``. Every scene has
    one ``sky`` region above a jittered horizon, one ``ground`` band below it, and 1–3 things (disc, box
    or triangle) of jittered size, position and colour; a thing drawn over another may hide it entirely,
    in which case the hidden instance is dropped, so instance counts vary from 3 to 5.
    """
    if not 1 <= n_images <= MAX_DATASET_IMAGES:
        raise ValueError(f"n_images {n_images} outside 1..MAX_DATASET_IMAGES {MAX_DATASET_IMAGES}")
    rng = np.random.default_rng(seed)
    records = []
    for index in range(n_images):
        tint = tuple(int(v) for v in rng.integers(225, 250, 3))
        image = Image.new("RGB", (width, height), tint)
        ids = Image.new("I", (width, height), 1)
        d, di = ImageDraw.Draw(image), ImageDraw.Draw(ids)
        horizon = int(rng.integers(int(height * 0.55), int(height * 0.75)))
        ground = (int(rng.integers(40, 90)), int(rng.integers(150, 200)), int(rng.integers(50, 100)))
        d.rectangle([0, horizon, width, height], fill=ground)
        di.rectangle([0, horizon, width, height], fill=2)
        instances: dict[int, str] = {1: "sky", 2: "ground"}
        next_id = 3
        for _ in range(int(rng.integers(1, 4))):
            name = SHAPE_CLASSES[int(rng.integers(2, len(SHAPE_CLASSES)))]
            side = int(rng.integers(40, 90))
            x0 = int(rng.integers(0, width - side))
            y0 = int(rng.integers(0, height - side))
            colour = tuple(int(v) for v in rng.integers(20, 235, 3))
            if name == "disc":
                kind, geometry = "ellipse", [x0, y0, x0 + side, y0 + side]
            elif name == "box":
                kind, geometry = "rectangle", [x0, y0, x0 + side, y0 + int(side * 0.8)]
            else:
                kind, geometry = "polygon", [(x0, y0 + side), (x0 + side // 2, y0), (x0 + side, y0 + side)]
            _shape(d, kind, geometry, colour)
            _shape(di, kind, geometry, next_id)
            instances[next_id] = name
            next_id += 1
        instance_map = np.array(ids, dtype=np.int32)
        present = {int(v) for v in np.unique(instance_map)}
        instances = {k: v for k, v in instances.items() if k in present}
        records.append(
            {
                "id": f"scene-{seed}-{index:03d}",
                "image": image,
                "instance_map": instance_map,
                "instances": instances,
            }
        )
    return records


def record_digest(record: Mapping[str, Any]) -> str:
    """SHA-256 over the RGB pixels, the instance map and the instance→class mapping of one record."""
    digest = hashlib.sha256()
    digest.update(np.asarray(record["image"].convert("RGB"), dtype=np.uint8).tobytes())
    digest.update(np.ascontiguousarray(record["instance_map"], dtype=np.int32).tobytes())
    digest.update(json.dumps({str(k): v for k, v in sorted(record["instances"].items())}).encode("utf-8"))
    return digest.hexdigest()


DATASET_SCHEMA: dict[str, Any] = {
    "record": (
        "{'id': str, 'image': PIL.Image.Image, 'instance_map': int32 array (H, W) with one id per pixel "
        "(0 is not allowed: every pixel belongs to an instance), 'instances': {instance_id: class_name}}"
    ),
    "class_names": f"{MIN_CLASSES}..{MAX_CLASSES} distinct names of at most {MAX_CLASS_NAME_CHARS} chars",
    "stuff_names": "subset of class_names whose regions are amorphous (fused per image); the rest are things",
    "images": [1, MAX_DATASET_IMAGES],
    "instances_per_image": [1, MAX_INSTANCES_PER_IMAGE],
    "instance_ids": [1, MAX_INSTANCE_ID],
    "epochs": [1, MAX_EPOCHS],
    "preprocessing": (
        "images are converted to RGB and resized by the pinned Mask2Former image processor (384x384, aspect "
        "ratio not preserved, ImageNet mean/std); instance maps are resized with nearest-neighbour sampling "
        "to the same size and turned into one binary mask + class label per instance"
    ),
}


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    stuff_names: Sequence[str] = (),
    *,
    epochs: int = 1,
) -> dict[str, Any]:
    """Validation stage for the adaptation path: raise on the first contract violation, otherwise return a
    dataset manifest (schema, class vocabulary, per-class instance counts, digests, findings, verdict).

    A class with no instance anywhere in ``records`` is not an error but a recorded finding: the head can
    be trained without it, yet nothing about that class will have been learned.
    """
    names = list(class_names)
    if not MIN_CLASSES <= len(names) <= MAX_CLASSES:
        raise ValueError(f"class count {len(names)} outside {MIN_CLASSES}..{MAX_CLASSES}")
    if len(set(names)) != len(names):
        raise ValueError("class names must be distinct")
    for name in names:
        if not isinstance(name, str) or not name.strip() or len(name) > MAX_CLASS_NAME_CHARS:
            raise ValueError(
                f"class name {name!r} must be a non-empty str of at most {MAX_CLASS_NAME_CHARS} chars"
            )
    stuff = list(stuff_names)
    unknown = [name for name in stuff if name not in names]
    if unknown:
        raise ValueError(f"stuff names {unknown} are not in class_names")
    if isinstance(records, Mapping) or not isinstance(records, Sequence):
        raise TypeError("records must be a sequence of record dicts")
    if not 1 <= len(records) <= MAX_DATASET_IMAGES:
        raise ValueError(f"record count {len(records)} outside 1..MAX_DATASET_IMAGES {MAX_DATASET_IMAGES}")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= MAX_EPOCHS:
        raise ValueError(f"epochs must be an int in 1..MAX_EPOCHS {MAX_EPOCHS}, got {epochs!r}")
    counts = dict.fromkeys(names, 0)
    ids: set[str] = set()
    digests: list[str] = []
    sizes: set[tuple[int, int]] = set()
    for position, record in enumerate(records):
        if not isinstance(record, Mapping) or not {"id", "image", "instance_map", "instances"} <= set(record):
            raise ValueError(f"record {position}: expected keys id, image, instance_map, instances")
        record_id = record["id"]
        if not isinstance(record_id, str) or not record_id:
            raise ValueError(f"record {position}: id must be a non-empty str")
        if record_id in ids:
            raise ValueError(f"record {position}: duplicate id {record_id!r}")
        ids.add(record_id)
        image = record["image"]
        if not isinstance(image, Image.Image):
            raise TypeError(f"record {record_id!r}: image must be a PIL.Image.Image")
        instance_map = np.asarray(record["instance_map"])
        if instance_map.ndim != 2 or not np.issubdtype(instance_map.dtype, np.integer):
            raise ValueError(f"record {record_id!r}: instance_map must be a 2-D integer array")
        if instance_map.shape != (image.height, image.width):
            raise ValueError(
                f"record {record_id!r}: instance_map shape {instance_map.shape} != image (H, W) "
                f"{(image.height, image.width)}"
            )
        present = {int(v) for v in np.unique(instance_map)}
        instances = record["instances"]
        if not isinstance(instances, Mapping) or not instances:
            raise ValueError(f"record {record_id!r}: instances must be a non-empty {{id: class}} mapping")
        if not 1 <= len(instances) <= MAX_INSTANCES_PER_IMAGE:
            raise ValueError(f"record {record_id!r}: {len(instances)} instances > MAX_INSTANCES_PER_IMAGE")
        keys = {int(k) for k in instances}
        if any(not 1 <= key <= MAX_INSTANCE_ID for key in keys):
            raise ValueError(
                f"record {record_id!r}: instance ids must lie in 1..MAX_INSTANCE_ID {MAX_INSTANCE_ID}"
            )
        if 0 in present or present != keys:
            raise ValueError(
                f"record {record_id!r}: instance ids in the map {sorted(present)} must equal the mapping's "
                f"{sorted(keys)} and contain no 0 (every pixel must belong to an instance)"
            )
        for key, name in instances.items():
            if name not in counts:
                raise ValueError(
                    f"record {record_id!r}: instance {key} has class {name!r} not in class_names"
                )
            counts[name] += 1
        sizes.add(image.size)
        digests.append(record_digest(record))
    findings = [
        {
            "class": name,
            "verdict": "no-instances",
            "message": f"class {name!r} has no instance in the dataset",
        }
        for name, count in counts.items()
        if count == 0
    ]
    return {
        "schema": dict(DATASET_SCHEMA),
        "class_names": names,
        "stuff_names": stuff,
        "thing_names": [name for name in names if name not in stuff],
        "n_records": len(records),
        "instances_per_class": counts,
        "image_sizes": sorted(sizes),
        "epochs": epochs,
        "dataset_sha256": hashlib.sha256("".join(digests).encode("ascii")).hexdigest(),
        "verdict": "accepted",
        "findings": findings,
    }


def split_records(
    records: Sequence[Mapping[str, Any]], holdout: float = 0.25, seed: int = 0
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Seeded random split into ``(train, held_out)``; every record is drawn independently so a random
    split is the right kind (SPL3). At least one record lands on each side."""
    if not 0.0 < holdout < 1.0:
        raise ValueError(f"holdout must be in (0, 1), got {holdout!r}")
    if len(records) < 2:
        raise ValueError("at least two records are needed to split")
    n_held = min(max(1, round(len(records) * holdout)), len(records) - 1)
    order = np.random.default_rng(seed).permutation(len(records))
    held = sorted(order[:n_held].tolist())
    train = sorted(order[n_held:].tolist())
    return [dict(records[i]) for i in train], [dict(records[i]) for i in held]


def load_labelled_dir(root: str | Path) -> tuple[list[dict[str, Any]], list[str], list[str]]:
    """Bring-your-own labelled dataset: ``<root>/dataset.json`` with ``class_names``, ``stuff_names`` and
    ``records`` (``{"id", "image", "mask", "instances"}``, paths relative to ``root``); each ``mask`` is a
    PNG whose pixel value is the instance id (mode ``I``, ``I;16`` or ``L``). Returns records ready for
    ``validate_dataset`` plus the two vocabularies."""
    root = Path(root)
    spec = json.loads((root / "dataset.json").read_text(encoding="utf-8"))
    records = []
    for entry in spec["records"]:
        image = Image.open(root / entry["image"])
        image.load()
        mask = Image.open(root / entry["mask"])
        instance_map = np.array(mask.convert("I"), dtype=np.int32)
        records.append(
            {
                "id": str(entry["id"]),
                "image": image,
                "instance_map": instance_map,
                "instances": {int(k): str(v) for k, v in entry["instances"].items()},
            }
        )
    return records, list(spec["class_names"]), list(spec.get("stuff_names", []))

**Module 2/2:** `src/mask2former_panoptic_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Panoptic segmentation with the pinned ``facebook/mask2former-swin-tiny-coco-panoptic`` checkpoint,
plus a bounded re-head-and-fine-tune path for a caller-supplied class vocabulary.

The class loads the image processor and model only from a digest-verified local snapshot
(``weights/<key>/``) or, when explicitly allowed, from the Hugging Face Hub at the pinned revision —
always with ``trust_remote_code=False``: the Mask2Former architecture comes from the pinned
``transformers`` release, the weights are SafeTensors, and no model-repository code is executed.

Panoptic post-processing is implemented here (NumPy + Pillow) following the upstream Mask2Former rule:
a query survives when its best class probability reaches ``score_threshold``; every pixel goes to the
surviving query with the highest score-weighted mask probability; a pixel is assigned only where that
query's own mask probability reaches ``mask_threshold``, otherwise it stays **void** (``-1``); a query
whose assigned area is below ``overlap_threshold`` of its thresholded mask is dropped; stuff queries of
the same class are fused into one segment. The pinned ``transformers`` post-processor differs in the
per-pixel rule (it assigns every pixel to the argmax query), which hands a whole image to a single
low-confidence query on out-of-domain input; the smoke run recorded that difference.
"""

from __future__ import annotations

import hashlib
import json
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

MODEL_ID = "facebook/mask2former-swin-tiny-coco-panoptic"
MODEL_REVISION = "df6b1142ff50c3276559d9d78f35f6a579c75a77"
MODEL_LICENSE = "mit"
MODEL_KEY = "mask2former-swin-tiny-coco-panoptic"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_MANIFEST_NAME = "dimer-adapted-manifest.json"
ARTIFACT_FORMAT = "dimer_mask2former_panoptic_adapted/1"

# The pinned processor resizes every image to INPUT_SIZE x INPUT_SIZE (aspect ratio not preserved) and
# the transformer decoder emits NUM_QUERIES mask logits at INPUT_SIZE / 4.
INPUT_SIZE = 384
MASK_LOGIT_SIZE = 96
NUM_QUERIES = 100
# COCO panoptic vocabulary of the checkpoint: label ids 0..79 are the 80 "thing" classes, 80..132 the
# 53 "stuff" classes (config.json id2label order).
COCO_THING_COUNT = 80
# Post-processing thresholds (caller-owned request parameters). SCORE_THRESHOLD is the pinned
# transformers default; the upstream Mask2Former panoptic inference uses 0.8 for the same test.
SCORE_THRESHOLD = 0.5
OVERLAP_THRESHOLD = 0.8
MASK_THRESHOLD = 0.5
# Input ceilings. Cost is bounded by the fixed resize; the side ceiling guards memory during the
# per-query resampling of masks to input resolution.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# Adaptation defaults (FT4/FT6): chosen for a practical CPU runtime, not inherited from the upstream
# recipe (which trains at 12,544 sampled points per mask for 50 epochs on COCO with a 1e-4 AdamW).
TRAIN_NUM_POINTS = 4096
DEFAULT_EPOCHS = 6
DEFAULT_BATCH_SIZE = 2
DEFAULT_LEARNING_RATE = 1e-4
DEFAULT_WEIGHT_DECAY = 0.05
MAX_BATCH_SIZE = 8


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


# --------------------------------------------------------------------------------------------------
# Input contract
# --------------------------------------------------------------------------------------------------


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_fraction(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one PIL.Image.Image (any mode, converted to RGB)",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "score_threshold": [0.0, 1.0],
    "overlap_threshold": [0.0, 1.0],
    "mask_threshold": [0.0, 1.0],
    "preprocessing": (
        f"image converted to RGB and resized to {INPUT_SIZE}x{INPUT_SIZE} (aspect ratio not preserved, "
        f"ImageNet mean/std); the decoder emits {NUM_QUERIES} (class distribution, {MASK_LOGIT_SIZE}x"
        f"{MASK_LOGIT_SIZE} mask logit) pairs; surviving masks are resampled bilinearly to the input size"
    ),
    "output": (
        "a panoptic map at input resolution (int32, one segment id per pixel, -1 = void) and one entry per "
        "segment: id, label, class index, thing/stuff, score (the query's softmax class probability, "
        "uncalibrated), area fraction, tight box, was_fused"
    ),
}


def _check_inputs(
    image: Any, score_threshold: Any, overlap_threshold: Any, mask_threshold: Any
) -> tuple[Image.Image, float, float, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``segment`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    return (
        rgb,
        _check_fraction("score_threshold", score_threshold),
        _check_fraction("overlap_threshold", overlap_threshold),
        _check_fraction("mask_threshold", mask_threshold),
    )


def validate_inputs(
    image: Image.Image,
    *,
    score_threshold: float = SCORE_THRESHOLD,
    overlap_threshold: float = OVERLAP_THRESHOLD,
    mask_threshold: float = MASK_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, score, overlap, mask = _check_inputs(image, score_threshold, overlap_threshold, mask_threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (segment takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "score_threshold": score,
        "overlap_threshold": overlap,
        "mask_threshold": mask,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


# --------------------------------------------------------------------------------------------------
# Panoptic post-processing (upstream rule, NumPy + Pillow) and panoptic quality
# --------------------------------------------------------------------------------------------------


def mask_bbox(mask: np.ndarray) -> list[int] | None:
    """Tight xyxy pixel box around the true pixels of a mask, or ``None`` for an empty mask."""
    rows = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=1))
    cols = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=0))
    if rows.size == 0 or cols.size == 0:
        return None
    return [int(cols[0]), int(rows[0]), int(cols[-1]) + 1, int(rows[-1]) + 1]


def _resample(mask_logit: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    """Bilinear resampling of one float32 map to ``(height, width)`` through Pillow's mode-F resize."""
    height, width = size
    if mask_logit.shape == (height, width):
        return np.asarray(mask_logit, dtype=np.float32)
    return np.asarray(Image.fromarray(np.asarray(mask_logit, dtype=np.float32)).resize((width, height)))


def panoptic_from_logits(
    class_logits: np.ndarray,
    mask_logits: np.ndarray,
    size: tuple[int, int],
    *,
    label_names: Sequence[str],
    stuff_ids: Sequence[int] | frozenset[int],
    score_threshold: float = SCORE_THRESHOLD,
    overlap_threshold: float = OVERLAP_THRESHOLD,
    mask_threshold: float = MASK_THRESHOLD,
) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """Turn one image's raw decoder outputs into a panoptic map and its segment list.

    ``class_logits`` is ``(NUM_QUERIES, K + 1)`` (last column = no object), ``mask_logits`` is
    ``(NUM_QUERIES, h, w)``; ``size`` is the ``(height, width)`` of the input image. Returns an int32
    ``(height, width)`` map whose value is the segment id (1-based) or ``-1`` for void, plus one dict per
    segment ordered by decreasing score. Stuff queries of the same class are fused into one segment.
    """
    class_logits = np.asarray(class_logits, dtype=np.float64)
    mask_logits = np.asarray(mask_logits, dtype=np.float32)
    if class_logits.ndim != 2 or mask_logits.ndim != 3 or class_logits.shape[0] != mask_logits.shape[0]:
        raise ValueError("class_logits must be (Q, K+1) and mask_logits (Q, h, w) with the same Q")
    n_classes = class_logits.shape[1] - 1
    if n_classes != len(label_names):
        raise ValueError(f"class_logits has {n_classes} classes, label_names has {len(label_names)}")
    stuff = frozenset(int(i) for i in stuff_ids)
    shifted = class_logits - class_logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)
    scores = probs[:, :-1].max(axis=1)
    labels = probs[:, :-1].argmax(axis=1)
    keep = np.flatnonzero(scores > score_threshold)
    height, width = size
    segmentation = np.full((height, width), -1, dtype=np.int32)
    if keep.size == 0:
        return segmentation, []
    keep = keep[np.argsort(-scores[keep], kind="stable")]
    mask_probs = np.stack([1.0 / (1.0 + np.exp(-_resample(mask_logits[q], size))) for q in keep])
    weighted = mask_probs * scores[keep][:, None, None].astype(np.float32)
    assignment = weighted.argmax(axis=0)
    segments: list[dict[str, Any]] = []
    stuff_segment: dict[int, int] = {}
    for position, query in enumerate(keep):
        label = int(labels[query])
        argmax_region = assignment == position
        confident = mask_probs[position] >= mask_threshold
        mask = argmax_region & confident
        mask_area = int(argmax_region.sum())
        original_area = int(confident.sum())
        if mask_area == 0 or original_area == 0 or not mask.any():
            continue
        if mask_area / original_area < overlap_threshold:
            continue
        if label in stuff and label in stuff_segment:
            segment_id = stuff_segment[label]
            segmentation[mask] = segment_id
            entry = next(entry for entry in segments if entry["id"] == segment_id)
            entry["was_fused"] = True
            entry["score"] = max(entry["score"], float(scores[query]))
            continue
        segment_id = len(segments) + 1
        segmentation[mask] = segment_id
        if label in stuff:
            stuff_segment[label] = segment_id
        segments.append(
            {
                "id": segment_id,
                "label": str(label_names[label]),
                "label_id": label,
                "is_thing": label not in stuff,
                "score": float(scores[query]),
                "query": int(query),
                "was_fused": False,
            }
        )
    for entry in segments:
        mask = segmentation == entry["id"]
        entry["area_fraction"] = float(mask.mean())
        entry["bbox"] = mask_bbox(mask)
    return segmentation, segments


def _segment_masks(segmentation: np.ndarray, segments: Sequence[Mapping[str, Any]]) -> list[np.ndarray]:
    return [np.asarray(segmentation) == int(entry["id"]) for entry in segments]


def panoptic_quality(
    pairs: Sequence[tuple[np.ndarray, Sequence[Mapping[str, Any]], np.ndarray, Sequence[Mapping[str, Any]]]],
    *,
    class_agnostic: bool = False,
) -> dict[str, Any]:
    """Panoptic quality (Kirillov et al., 2019) over one or more ``(pred_seg, pred_segments, ref_seg,
    ref_segments)`` pairs.

    Segments are matched per class (or regardless of class when ``class_agnostic``) when their IoU exceeds
    0.5 — a rule that makes matches unique. Reference void pixels (``-1``) are ignored: they are removed
    from predicted masks before IoU, and a predicted segment lying mostly (> 50 %) on void is neither a
    match nor a false positive. Returns ``pq``, ``sq``, ``rq`` overall and per class, thing/stuff
    aggregates, and the raw ``tp`` / ``fp`` / ``fn`` / ``iou_sum`` counts they were computed from
    (per-class values are the means over classes that occur in the references or the predictions).
    """
    totals: dict[str, dict[str, float]] = {}
    thing_flag: dict[str, bool] = {}

    def bucket(entry: Mapping[str, Any]) -> str:
        return "region" if class_agnostic else str(entry.get("label", entry.get("name")))

    for pred_seg, pred_segments, ref_seg, ref_segments in pairs:
        pred_seg, ref_seg = np.asarray(pred_seg), np.asarray(ref_seg)
        if pred_seg.shape != ref_seg.shape:
            raise ValueError(f"segmentation shapes differ: {pred_seg.shape} vs {ref_seg.shape}")
        void = ref_seg == -1
        ref_masks = _segment_masks(ref_seg, ref_segments)
        pred_masks = [mask & ~void for mask in _segment_masks(pred_seg, pred_segments)]
        pred_void_fraction = [
            float((mask & void).sum() / mask.sum()) if mask.any() else 0.0
            for mask in _segment_masks(pred_seg, pred_segments)
        ]
        for entry in list(ref_segments) + list(pred_segments):
            key = bucket(entry)
            totals.setdefault(key, {"iou_sum": 0.0, "tp": 0, "fp": 0, "fn": 0})
            thing_flag.setdefault(key, bool(entry.get("is_thing", True)))
        matched_pred: set[int] = set()
        for ref_index, ref_entry in enumerate(ref_segments):
            key = bucket(ref_entry)
            best_iou, best_pred = 0.0, -1
            for pred_index, pred_entry in enumerate(pred_segments):
                if pred_index in matched_pred or bucket(pred_entry) != key:
                    continue
                inter = np.logical_and(ref_masks[ref_index], pred_masks[pred_index]).sum()
                if inter == 0:
                    continue
                union = np.logical_or(ref_masks[ref_index], pred_masks[pred_index]).sum()
                iou = float(inter / union)
                if iou > best_iou:
                    best_iou, best_pred = iou, pred_index
            if best_iou > 0.5:
                matched_pred.add(best_pred)
                totals[key]["tp"] += 1
                totals[key]["iou_sum"] += best_iou
            else:
                totals[key]["fn"] += 1
        for pred_index, pred_entry in enumerate(pred_segments):
            if pred_index in matched_pred or pred_void_fraction[pred_index] > 0.5:
                continue
            totals[bucket(pred_entry)]["fp"] += 1

    def summarise(counts: Mapping[str, float]) -> dict[str, float]:
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        denominator = tp + 0.5 * fp + 0.5 * fn
        sq = counts["iou_sum"] / tp if tp else 0.0
        rq = tp / denominator if denominator else 0.0
        return {"pq": sq * rq, "sq": sq, "rq": rq, "tp": int(tp), "fp": int(fp), "fn": int(fn)}

    per_class = {key: summarise(counts) for key, counts in totals.items()}

    def mean_over(keys: Sequence[str]) -> dict[str, float]:
        if not keys:
            return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "n_classes": 0}
        return {
            **{
                metric: float(np.mean([per_class[key][metric] for key in keys]))
                for metric in ("pq", "sq", "rq")
            },
            "n_classes": len(keys),
        }

    keys = sorted(per_class)
    return {
        "pq": mean_over(keys)["pq"],
        "sq": mean_over(keys)["sq"],
        "rq": mean_over(keys)["rq"],
        "n_classes": len(keys),
        "things": mean_over([key for key in keys if thing_flag[key]]),
        "stuff": mean_over([key for key in keys if not thing_flag[key]]),
        "per_class": per_class,
        "class_agnostic": class_agnostic,
        "n_images": len(pairs),
        "matching": "IoU > 0.5 per class (unique by construction); reference void pixels ignored",
    }


def reference_from_record(record: Mapping[str, Any], class_names: Sequence[str], stuff_names: Sequence[str]):
    """Turn a dataset record into ``(segmentation, segments)`` in the panoptic_quality convention."""
    instance_map = np.asarray(record["instance_map"], dtype=np.int32)
    segments = []
    for instance_id, name in sorted(record["instances"].items(), key=lambda item: int(item[0])):
        mask = instance_map == int(instance_id)
        segments.append(
            {
                "id": int(instance_id),
                "label": name,
                "label_id": list(class_names).index(name),
                "is_thing": name not in stuff_names,
                "area_fraction": float(mask.mean()),
            }
        )
    return instance_map, segments


def evaluation_report(
    result: Mapping[str, Any],
    reference: tuple[np.ndarray, Sequence[Mapping[str, Any]]] | None = None,
    *,
    sample_kind: str = "synthetic",
    class_agnostic: bool = True,
) -> dict[str, Any]:
    """Evaluation stage for the inference path: a machine-readable report even when nothing is measurable.

    With ``reference`` (a panoptic map and its segment list, e.g. the regions a drawn scene was drawn
    from) the report carries panoptic quality — class-agnostic by default, because the checkpoint's COCO
    vocabulary does not name drawn regions — as ``sample-sanity`` evidence; without it the verdict is
    ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    segments = list(result["segments"])
    base = {
        "task": "panoptic segmentation: every pixel is assigned to one segment (thing instance or fused"
        " stuff) or void",
        "decision_rule": (
            "a query survives when its best class probability reaches score_threshold; each pixel goes to "
            "the surviving query with the highest score-weighted mask probability and is assigned only where "
            "that query's mask probability reaches mask_threshold (otherwise void); a query whose assigned "
            "area is below overlap_threshold of its thresholded mask is dropped; scores are uncalibrated "
            "softmaxes"
        ),
        "thresholds": {
            "score_threshold": result.get("score_threshold", SCORE_THRESHOLD),
            "overlap_threshold": result.get("overlap_threshold", OVERLAP_THRESHOLD),
            "mask_threshold": result.get("mask_threshold", MASK_THRESHOLD),
        },
        "sample_kind": sample_kind,
        "n_segments": len(segments),
        "void_fraction": float((np.asarray(result["segmentation"]) == -1).mean()),
        "labels": [
            (entry["label"], round(entry["score"], 4), round(entry["area_fraction"], 4)) for entry in segments
        ],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference panoptic map was supplied for the evaluated image",
            "needs": (
                "panoptic annotations (an instance map and a class per instance) on images from the "
                "deployment domain with a class vocabulary matching the model's, scored with panoptic "
                "quality (PQ = SQ x RQ, IoU > 0.5 matching); no such labelled set ships with this repository"
            ),
        }
    ref_seg, ref_segments = reference
    quality = panoptic_quality(
        [(np.asarray(result["segmentation"]), segments, np.asarray(ref_seg), list(ref_segments))],
        class_agnostic=class_agnostic,
    )
    estimation = "one reference map on a single image, no dispersion estimate"
    metrics = [
        {"id": "pq", "value": quality["pq"], "estimation": estimation},
        {"id": "sq", "value": quality["sq"], "estimation": estimation},
        {"id": "rq", "value": quality["rq"], "estimation": estimation},
        {
            "id": "matches",
            "value": {
                key: {k: v for k, v in counts.items() if k in ("tp", "fp", "fn")}
                for key, counts in quality["per_class"].items()
            },
            "estimation": "true/false positive and false negative segment counts behind pq",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "panoptic_quality": quality,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(ref_segments)} reference region(s) on one tutorial sample, "
            + ("matched regardless of class label" if class_agnostic else "matched per class")
            + "; geometry sanity evidence, not a COCO panoptic benchmark"
        ),
        "needs": "panoptic annotations from the deployment domain with a matching vocabulary for any PQ "
        "claim",
    }


# --------------------------------------------------------------------------------------------------
# The pipeline
# --------------------------------------------------------------------------------------------------


def _build_class_lists(config: Any) -> tuple[list[str], frozenset[int]]:
    names = [config.id2label[i] for i in range(len(config.id2label))]
    return names, frozenset(range(COCO_THING_COUNT, len(names)))


@dataclass
class Mask2FormerPanopticPipeline:
    """Panoptic segmentation over the pinned Mask2Former Swin-T COCO checkpoint, with an optional
    re-headed class vocabulary that ``finetune`` adapts on caller-supplied labelled scenes."""

    model: Any
    processor: Any
    device: str
    class_names: tuple[str, ...]
    stuff_ids: frozenset[int]
    source: str
    adapted: bool = False
    reinitialised: list[str] = field(default_factory=list)
    seed: int | None = None
    training: dict[str, Any] | None = None

    @property
    def stuff_names(self) -> tuple[str, ...]:
        return tuple(self.class_names[i] for i in sorted(self.stuff_ids))

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        *,
        class_names: Sequence[str] | None = None,
        stuff_names: Sequence[str] = (),
        seed: int = 0,
        train_num_points: int = TRAIN_NUM_POINTS,
    ) -> Mask2FormerPanopticPipeline:
        """Load the verified snapshot. With ``class_names`` the classification head is rebuilt for that
        vocabulary (seeded random initialisation of exactly the head; everything else transfers) so the
        model can be adapted with ``finetune``; ``stuff_names`` marks the amorphous classes."""
        if class_names is not None:
            names = list(class_names)
            if len(names) < 2 or len(set(names)) != len(names):
                raise ValueError("class_names must hold at least two distinct names")
            unknown = [name for name in stuff_names if name not in names]
            if unknown:
                raise ValueError(f"stuff names {unknown} are not in class_names")
        elif stuff_names:
            raise ValueError("stuff_names needs class_names (the COCO vocabulary carries its own stuff set)")
        if (
            isinstance(train_num_points, bool)
            or not isinstance(train_num_points, int)
            or train_num_points < 1
        ):
            raise ValueError(f"train_num_points must be a positive int, got {train_num_points!r}")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        head: dict[str, Any] = {"train_num_points": train_num_points}
        if class_names is not None:
            head.update(
                num_labels=len(names),
                id2label=dict(enumerate(names)),
                label2id={name: index for index, name in enumerate(names)},
                ignore_mismatched_sizes=True,
            )
            torch.manual_seed(seed)
        model, info = Mask2FormerForUniversalSegmentation.from_pretrained(
            source,
            revision=MODEL_REVISION,
            trust_remote_code=False,
            dtype=torch.float32,
            output_loading_info=True,
            **head,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        if class_names is not None:
            # mismatched_keys entries are key names (pinned transformers) or (key, old, new) tuples (older).
            reinitialised = sorted(
                {entry if isinstance(entry, str) else entry[0] for entry in info.get("mismatched_keys", [])}
            )
            stuff = frozenset(names.index(name) for name in stuff_names)
            return cls(
                model, processor, resolved_device, tuple(names), stuff, source, False, reinitialised, seed
            )
        names, stuff = _build_class_lists(model.config)
        return cls(model, processor, resolved_device, tuple(names), stuff, source)

    # -- inference ---------------------------------------------------------------------------------

    def _raw(self, rgb: Image.Image) -> tuple[np.ndarray, np.ndarray]:
        import torch

        inputs = self.processor(images=rgb, return_tensors="pt").to(self.device)
        self.model.eval()
        with torch.inference_mode():
            outputs = self.model(pixel_values=inputs["pixel_values"], pixel_mask=inputs.get("pixel_mask"))
        return (
            outputs.class_queries_logits[0].float().cpu().numpy(),
            outputs.masks_queries_logits[0].float().cpu().numpy(),
        )

    def segment(
        self,
        image: Image.Image,
        *,
        score_threshold: float = SCORE_THRESHOLD,
        overlap_threshold: float = OVERLAP_THRESHOLD,
        mask_threshold: float = MASK_THRESHOLD,
    ) -> dict[str, Any]:
        """Panoptic-segment one image; the map and the segment list are at input resolution."""
        rgb, score, overlap, mask = _check_inputs(image, score_threshold, overlap_threshold, mask_threshold)
        class_logits, mask_logits = self._raw(rgb)
        if (
            class_logits.shape != (NUM_QUERIES, len(self.class_names) + 1)
            or mask_logits.shape[0] != NUM_QUERIES
        ):
            raise RuntimeError(
                f"backend returned class logits {class_logits.shape} and mask logits {mask_logits.shape}, "
                f"expected ({NUM_QUERIES}, {len(self.class_names) + 1}) and ({NUM_QUERIES}, h, w)"
            )
        segmentation, segments = panoptic_from_logits(
            class_logits,
            mask_logits,
            (rgb.height, rgb.width),
            label_names=self.class_names,
            stuff_ids=self.stuff_ids,
            score_threshold=score,
            overlap_threshold=overlap,
            mask_threshold=mask,
        )
        return {
            "segmentation": segmentation,
            "segments": segments,
            "void_fraction": float((segmentation == -1).mean()),
            "width": rgb.width,
            "height": rgb.height,
            "score_threshold": score,
            "overlap_threshold": overlap,
            "mask_threshold": mask,
            "class_names": list(self.class_names),
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # -- adaptation --------------------------------------------------------------------------------

    def _encode(self, records: Sequence[Mapping[str, Any]]) -> Any:
        label_index = {name: index for index, name in enumerate(self.class_names)}
        return self.processor(
            images=[validate_image(record["image"]) for record in records],
            segmentation_maps=[np.asarray(record["instance_map"], dtype=np.int32) for record in records],
            instance_id_to_semantic_id=[
                {int(key): label_index[name] for key, name in record["instances"].items()}
                for record in records
            ],
            return_tensors="pt",
        )

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        weight_decay: float = DEFAULT_WEIGHT_DECAY,
        seed: int = 0,
        freeze_backbone: bool = True,
        freeze_pixel_decoder: bool = False,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded gradient fine-tune on validated records with the upstream Mask2Former loss (Hungarian
        matching of queries to instances; class cross-entropy + point-sampled mask BCE and dice, weights
        2 / 5 / 5, no-object weight 0.1, auxiliary decoder losses on). Runs in place; returns the run record
        (hyperparameters, trainable parameter counts, per-epoch mean loss)."""
        manifest = validate_dataset(records, self.class_names, self.stuff_names, epochs=epochs)
        if (
            isinstance(batch_size, bool)
            or not isinstance(batch_size, int)
            or not 1 <= batch_size <= MAX_BATCH_SIZE
        ):
            raise ValueError(
                f"batch_size must be an int in 1..MAX_BATCH_SIZE {MAX_BATCH_SIZE}, got {batch_size!r}"
            )
        if not isinstance(learning_rate, int | float) or not 0.0 < learning_rate <= 1.0:
            raise ValueError(f"learning_rate must be in (0, 1], got {learning_rate!r}")
        import torch

        for name, parameter in self.model.named_parameters():
            frozen = (freeze_backbone and name.startswith("model.pixel_level_module.encoder")) or (
                freeze_pixel_decoder and name.startswith("model.pixel_level_module.decoder")
            )
            parameter.requires_grad_(not frozen)
        trainable = [parameter for parameter in self.model.parameters() if parameter.requires_grad]
        n_trainable = sum(parameter.numel() for parameter in trainable)
        n_total = sum(parameter.numel() for parameter in self.model.parameters())
        optimizer = torch.optim.AdamW(trainable, lr=learning_rate, weight_decay=weight_decay)
        history: list[dict[str, Any]] = []
        started = time.time()
        step = 0
        for epoch in range(1, epochs + 1):
            self.model.train()
            if freeze_backbone:
                self.model.model.pixel_level_module.encoder.eval()
            order = np.random.default_rng(seed + epoch).permutation(len(records)).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                batch = [records[index] for index in order[start : start + batch_size]]
                encoded = self._encode(batch)
                outputs = self.model(
                    pixel_values=encoded["pixel_values"].to(self.device),
                    pixel_mask=encoded["pixel_mask"].to(self.device),
                    mask_labels=[mask.to(self.device) for mask in encoded["mask_labels"]],
                    class_labels=[label.to(self.device) for label in encoded["class_labels"]],
                )
                outputs.loss.backward()
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                losses.append(float(outputs.loss.detach().cpu()))
                step += 1
            row = {
                "epoch": epoch,
                "steps": step,
                "mean_loss": float(np.mean(losses)),
                "last_loss": losses[-1],
                "seconds": round(time.time() - started, 1),
            }
            history.append(row)
            if progress is not None:
                progress(row)
        self.model.eval()
        self.adapted = True
        self.training = {
            "adaptation": "gradient fine-tuning of the transformer decoder"
            + ("" if freeze_pixel_decoder else " and pixel decoder")
            + " with a re-initialised class head; backbone "
            + ("frozen" if freeze_backbone else "trainable"),
            "loss": "Mask2Former set loss: Hungarian matching; class CE (2.0) + point-sampled mask BCE (5.0) "
            "+ dice (5.0); no-object weight 0.1; auxiliary decoder losses",
            "optimizer": "AdamW",
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "epochs": epochs,
            "batch_size": batch_size,
            "steps": step,
            "seed": seed,
            "precision": "float32",
            "train_num_points": int(self.model.config.train_num_points),
            "freeze_backbone": freeze_backbone,
            "freeze_pixel_decoder": freeze_pixel_decoder,
            "trainable_parameters": n_trainable,
            "total_parameters": n_total,
            "n_records": len(records),
            "dataset_sha256": manifest["dataset_sha256"],
            "history": history,
            "seconds": round(time.time() - started, 1),
            "device": self.device,
        }
        return dict(self.training)

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        score_threshold: float = SCORE_THRESHOLD,
        overlap_threshold: float = OVERLAP_THRESHOLD,
        mask_threshold: float = MASK_THRESHOLD,
    ) -> dict[str, Any]:
        """Class-aware panoptic quality of the current weights over labelled records (the model's own
        vocabulary must be the records' vocabulary); the same ``segment`` call as inference."""
        validate_dataset(records, self.class_names, self.stuff_names)
        pairs = []
        per_image = []
        for record in records:
            result = self.segment(
                record["image"],
                score_threshold=score_threshold,
                overlap_threshold=overlap_threshold,
                mask_threshold=mask_threshold,
            )
            ref_seg, ref_segments = reference_from_record(record, self.class_names, self.stuff_names)
            pairs.append((result["segmentation"], result["segments"], ref_seg, ref_segments))
            per_image.append(
                {
                    "id": record["id"],
                    "n_predicted": len(result["segments"]),
                    "n_reference": len(ref_segments),
                    "void_fraction": result["void_fraction"],
                }
            )
        quality = panoptic_quality(pairs)
        return {
            **quality,
            "per_image": per_image,
            "score_threshold": score_threshold,
            "overlap_threshold": overlap_threshold,
            "mask_threshold": mask_threshold,
            "adapted": self.adapted,
            "estimation": "held-out records scored once with the request thresholds; no dispersion estimate",
        }

    # -- artifact ----------------------------------------------------------------------------------

    def save_artifact(self, path: str | Path, *, notes: str = "") -> dict[str, Any]:
        """Export the adapted model as a self-describing directory: the Transformers ``config.json`` (with the
        class vocabulary) and ``model.safetensors``, the image processor configuration, and
        ``dimer-adapted-manifest.json`` naming the format, the base identity and the digest of every file."""
        if not self.adapted:
            raise ValueError("save_artifact needs an adapted model; call finetune first")
        root = Path(path)
        root.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(root, safe_serialization=True)
        self.processor.save_pretrained(root)
        files = sorted(
            entry.name for entry in root.iterdir() if entry.is_file() and entry.name != ARTIFACT_MANIFEST_NAME
        )
        manifest = {
            "format": ARTIFACT_FORMAT,
            "base": {"modelId": MODEL_ID, "revision": MODEL_REVISION, "license": MODEL_LICENSE},
            "class_names": list(self.class_names),
            "stuff_names": list(self.stuff_names),
            "head_seed": self.seed,
            "training": self.training,
            "notes": notes,
            "files": [
                {"path": name, "bytes": (root / name).stat().st_size, "sha256": _sha256(root / name)}
                for name in files
            ],
        }
        with open(root / ARTIFACT_MANIFEST_NAME, "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, indent=2)
        return {
            "path": str(root),
            "format": ARTIFACT_FORMAT,
            "files": manifest["files"],
            "class_names": manifest["class_names"],
            "stuff_names": manifest["stuff_names"],
            "total_bytes": sum(entry["bytes"] for entry in manifest["files"]),
        }

    @classmethod
    def load_artifact(cls, path: str | Path, device: str | None = None) -> Mask2FormerPanopticPipeline:
        """Fresh reload from an exported directory: refuses a foreign format or base identity and any
        file whose digest differs from the artifact manifest before importing model libraries."""
        root = Path(path)
        manifest_path = root / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as fh:
            manifest = json.load(fh)
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base", {})
        if base.get("modelId") != MODEL_ID or base.get("revision") != MODEL_REVISION:
            raise ValueError(f"artifact base {base} != package pins {MODEL_ID}@{MODEL_REVISION}")
        for entry in manifest["files"]:
            file_path = root / entry["path"]
            if not file_path.is_file():
                raise FileNotFoundError(f"artifact file missing: {file_path}")
            if file_path.stat().st_size != entry["bytes"] or _sha256(file_path) != entry["sha256"]:
                raise ValueError(f"{entry['path']}: digest differs from the artifact manifest")
        names = list(manifest["class_names"])
        stuff = frozenset(names.index(name) for name in manifest.get("stuff_names", []))
        import torch
        from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = AutoImageProcessor.from_pretrained(root, trust_remote_code=False, local_files_only=True)
        model = Mask2FormerForUniversalSegmentation.from_pretrained(
            root, trust_remote_code=False, local_files_only=True, dtype=torch.float32
        )
        loaded = [model.config.id2label[i] for i in range(len(model.config.id2label))]
        if loaded != names:
            raise ValueError(f"artifact config vocabulary {loaded} != manifest {names}")
        model = model.to(resolved_device).eval()
        return cls(
            model,
            processor,
            resolved_device,
            tuple(names),
            stuff,
            str(root),
            True,
            [],
            manifest.get("head_seed"),
            manifest.get("training"),
        )

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `df6b1142ff50…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Mask2FormerPanopticPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "mask2former-swin-tiny-coco-panoptic",
  "modelId": "facebook/mask2former-swin-tiny-coco-panoptic",
  "revision": "df6b1142ff50c3276559d9d78f35f6a579c75a77",
  "files": [
    {
      "path": "README.md",
      "bytes": 3190,
      "sha256": "8cf3afebeb22ca83bf13376d55c7781d3c96627bb4532de819ae3cec0ae303e9"
    },
    {
      "path": "config.json",
      "bytes": 82185,
      "sha256": "c638e5e4aeada33f37287509785306277cb6b60998cdd9dd13e9df7f0aa90055"
    },
    {
      "path": "model.safetensors",
      "bytes": 190052872,
      "sha256": "8e154c4f946f4fad9aa511a7d706c89c4254cc6eee28fd932e52c32394c5a29f"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 538,
      "sha256": "c25764d27c09d4cd2dcb0100f54221c00c7169446425cc70b2b40481157c3e5c"
    }
  ],
  "totalBytes": 190138785
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Mask2FormerPanopticPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the COCO checkpoint does on a drawn scene

Before adapting anything, see the starting point. `pipe` (Section 3) is the unmodified checkpoint with its 133 COCO categories. On `synthetic_scene` — a red disc, a blue box, a yellow triangle, a green ground band, an off-white sky — the repository's smoke run kept a single query, `stop sign` at score 0.601 on the disc, and left 91.7 % of the pixels void. That is not a bug: the checkpoint has no class for a drawn disc and the carried post-processing keeps void where no query is confident. The regions drawn here are the vocabulary the rest of the notebook teaches the model. Nothing is scored in this cell; it is the observation the adaptation is measured against.

In [ ]:
import json
import os
import time

import numpy as np
from PIL import Image

os.makedirs('outputs', exist_ok=True)
scene_image, scene_segmentation, scene_segments = synthetic_scene()
t0 = time.time()
coco_result = pipe.segment(scene_image)
coco_observation = {'seconds': round(time.time() - t0, 2), 'vocabulary_size': len(pipe.class_names),
                    'segments': [(s['label'], round(s['score'], 3), round(s['area_fraction'], 3)) for s in coco_result['segments']],
                    'void_fraction': round(coco_result['void_fraction'], 3),
                    'drawn_regions': [(s['name'], 'stuff' if not s['is_thing'] else 'thing', round(s['area_fraction'], 3)) for s in scene_segments]}
print(json.dumps(coco_observation, indent=2))
scene_image

## 5. Sample data for adaptation, and the validation stage

`shape_dataset(N_IMAGES, seed=DATASET_SEED)` draws a deterministic labelled set over `SHAPE_CLASSES` — `sky` and `ground` as **stuff** (one amorphous region each), `disc`, `box` and `triangle` as **things** (1–3 per scene, jittered size, position and colour) — and returns exact instance maps because it knows where it drew. A thing drawn over another can hide it entirely; hidden instances are dropped, so scenes carry 3–5 instances.

`validate_dataset` is the validation stage for this path and applies exactly the ceilings `finetune` applies, so a dataset it accepts cannot be refused later: record shape, instance map geometry (every pixel assigned, ids in 1..254, map and mapping agree), labels drawn from the vocabulary, at most 50 instances per image, at most 200 images, at most 20 epochs. It returns a **dataset manifest** — schema, vocabulary, per-class instance counts, a digest of the whole set — and reports a class with no instances as a **finding** rather than an error. The cell also validates a deliberately broken record (an instance map with an unassigned pixel) and records the pipeline's own rejection. The manifest is written to `outputs/mask2former_panoptic_finetune_dataset_manifest.json`.

In [ ]:
N_IMAGES = 24  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 6  # @param {type:"integer"}

records = shape_dataset(N_IMAGES, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, SHAPE_CLASSES, SHAPE_STUFF, epochs=EPOCHS)
# Demonstrate rejection: an instance map with an unassigned (0) pixel breaks the contract.
broken = dict(records[0])
broken['instance_map'] = records[0]['instance_map'].copy()
broken['instance_map'][0, 0] = 0
try:
    validate_dataset([broken], SHAPE_CLASSES, SHAPE_STUFF)
except ValueError as exc:
    dataset_manifest['findings'].append({'record': 'unassigned-pixel-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/mask2former_panoptic_finetune_dataset_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(dataset_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in dataset_manifest.items() if k != 'schema'}, indent=2))
print('record shape expected by finetune:', dataset_manifest['schema']['record'])
preview = Image.new('RGB', (960, 480))
for index, record in enumerate(records[:6]):
    preview.paste(record['image'], (320 * (index % 3), 240 * (index // 3)))
print('first six scenes; instances of the first:', records[0]['instances'])
preview

## 6. Split, re-head, and measure the baselines *before* adapting

`split_records` is a seeded permutation into a training part and a held-out part; every scene is drawn independently, so a random split is the right kind (SPL3). The split happens before any weight is touched and the held-out records are never shown to `finetune`, so the score in Section 8 is a score on scenes the adapted model has not seen. The 6 held-out scenes are used once, to report; nothing is selected on them (SPL6/SPL7).

`Mask2FormerPanopticPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SHAPE_CLASSES, stuff_names=SHAPE_STUFF, seed=SEED)` loads the same verified snapshot (no download: the files are already staged and re-verified) but rebuilds the class predictor for the five-class vocabulary. It records exactly which tensors could not transfer — `class_predictor.weight`/`.bias` and the loss's `criterion.empty_weight` — and everything else comes from the COCO checkpoint. The head's random initialisation is seeded, because it is the only untrained part of the model and precisely what the baseline measures.

Two baselines are recorded. The **re-headed model before adaptation** scored PQ 0.0 in the smoke run: the mask branch still proposes plausible regions, but a random class head assigns them arbitrary classes, and PQ is class-aware. The **trivial predictor** that labels the whole image `sky` (the majority class) scored PQ 0.10 (0.52 on `sky`, 0 elsewhere) — the number any adapted model must clear to have learned anything (EVAL10).

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_records(records, holdout=HOLDOUT, seed=SEED)
print({'train': len(train_records), 'held_out': len(held_out), 'train_instances': sum(len(r['instances']) for r in train_records), 'held_out_instances': sum(len(r['instances']) for r in held_out)})

adapter = Mask2FormerPanopticPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SHAPE_CLASSES, stuff_names=SHAPE_STUFF, seed=SEED)
print({'class_names': list(adapter.class_names), 'stuff_names': list(adapter.stuff_names), 'device': adapter.device, 'adapted': adapter.adapted, 'train_num_points': TRAIN_NUM_POINTS})
print('tensors re-initialised because they could not transfer from the COCO checkpoint:', adapter.reinitialised)

t0 = time.time()
baseline = adapter.evaluate(held_out)
print({'baseline_pq': round(baseline['pq'], 4), 'sq': round(baseline['sq'], 4), 'rq': round(baseline['rq'], 4), 'things_pq': round(baseline['things']['pq'], 4), 'stuff_pq': round(baseline['stuff']['pq'], 4), 'seconds': round(time.time() - t0, 1)})

# Trivial baseline: every pixel is the majority stuff class.
trivial_pairs = []
for record in held_out:
    reference_map, reference_segments = reference_from_record(record, SHAPE_CLASSES, SHAPE_STUFF)
    trivial_pairs.append((np.ones_like(reference_map), [{'id': 1, 'label': 'sky', 'is_thing': False}], reference_map, reference_segments))
trivial = panoptic_quality(trivial_pairs)
print({'trivial_all_sky_pq': round(trivial['pq'], 4), 'per_class': {k: round(v['pq'], 3) for k, v in trivial['per_class'].items()}})

## 7. The bounded fine-tune

This is the cell that makes the notebook an `E2E` tutorial rather than an inference demo, and it runs in the default path. The loss is **upstream's**, inside the pinned `transformers` Mask2Former: Hungarian matching of the 100 queries to the scene's instances, then class cross-entropy (weight 2.0), point-sampled mask binary cross-entropy (5.0) and dice (5.0) with a 0.1 weight on *no object*, plus the same loss on every auxiliary decoder layer. What the carried module owns is the bounded loop around it: the processor call that turns instance maps into per-instance masks and labels, batching, AdamW, the seed and the ceilings.

Defaults worth understanding, all explicit (FT4/FT6): `TRAIN_NUM_POINTS = 4096` sampled points per mask instead of the upstream 12,544 (a CPU-time decision recorded in the artifact); learning rate 1e-4 and weight decay 0.05 (the upstream optimizer values); batch size 2; 6 epochs over 18 scenes = 54 steps; float32; **the Swin-T backbone is frozen** and kept in eval mode, so 19.9 M of the 47.4 M parameters train (the pixel decoder, the transformer decoder and the new head). The smoke run's mean epoch loss went 33.7 → 6.8. **The loss history is optimisation evidence only** (FT7): whether the model learned the task is Section 8's question.

In [ ]:
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
FREEZE_BACKBONE = True  # @param {type:"boolean"}

run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_backbone=FREEZE_BACKBONE,
    progress=lambda row: print(f"epoch {row['epoch']}/{EPOCHS}  steps {row['steps']}  mean loss {row['mean_loss']:.3f}  last {row['last_loss']:.3f}  {row['seconds']:.0f} s"),
)
print(json.dumps({k: run[k] for k in ('adaptation', 'loss', 'optimizer', 'learning_rate', 'weight_decay', 'epochs', 'batch_size', 'steps', 'seed', 'precision', 'train_num_points', 'freeze_backbone', 'freeze_pixel_decoder', 'trainable_parameters', 'total_parameters', 'seconds', 'device')}, indent=2))
first, last = run['history'][0]['mean_loss'], run['history'][-1]['mean_loss']
print(f'mean epoch loss {first:.3f} -> {last:.3f} over {run["epochs"]} epochs (optimisation evidence only)')

## 8. Evaluate on the held-out split → evaluation report

The same `evaluate` call as the baseline, on the same held-out records, with the same thresholds — so the numbers are comparable and the only thing that changed is the weights. `evaluate` runs `segment` (the inference path, EVAL8) on every scene and accumulates **panoptic quality** per class: segments are matched at IoU > 0.5, `SQ` is the mean IoU of the matches, `RQ` is `TP / (TP + FP/2 + FN/2)`, and `PQ = SQ × RQ`, averaged over the five classes and reported for things and stuff separately. The estimation procedure is a single seeded holdout (EVAL5); there is no dispersion estimate.

What this number is: evidence that a bounded fine-tune on 18 drawn scenes moved a held-out score from 0.0 (and past the trivial 0.10). What it is not: a benchmark. The held-out split is six scenes from the same generator with the same three shapes, so a high PQ says the task is easy and the adaptation worked — not that the model would segment your photographs. The smoke run reached PQ 0.866 (SQ 0.938, RQ 0.911; stuff 0.999, things 0.777 with `box` the weakest at 0.509 and `triangle` at 0.981). The report is written to `outputs/mask2former_panoptic_finetune_evaluation_report.json`.

In [ ]:
t0 = time.time()
adapted = adapter.evaluate(held_out)
print({'adapted_pq': round(adapted['pq'], 4), 'sq': round(adapted['sq'], 4), 'rq': round(adapted['rq'], 4), 'things_pq': round(adapted['things']['pq'], 4), 'stuff_pq': round(adapted['stuff']['pq'], 4), 'seconds': round(time.time() - t0, 1)})
print('per class:', {k: {m: round(v[m], 3) for m in ('pq', 'sq', 'rq')} | {m: v[m] for m in ('tp', 'fp', 'fn')} for k, v in adapted['per_class'].items()})
print()
print(f"{'metric':<10s} {'trivial':>10s} {'baseline':>10s} {'adapted':>10s} {'change':>10s}")
for key in ('pq', 'sq', 'rq'):
    print(f'{key:<10s} {trivial[key]:>10.4f} {baseline[key]:>10.4f} {adapted[key]:>10.4f} {adapted[key] - baseline[key]:>+10.4f}')
evaluation = {'task': 'panoptic segmentation over the five-class drawn vocabulary', 'metric': 'panoptic quality (PQ = SQ x RQ), IoU > 0.5 matching, class-aware, mean over classes',
              'estimation': adapted['estimation'], 'split': {'train': len(train_records), 'held_out': len(held_out), 'holdout': HOLDOUT, 'seed': SEED},
              'baselines': [{'id': 'trivial-all-sky', 'pq': trivial['pq']}, {'id': 're-headed-before-adaptation', 'pq': baseline['pq']}],
              'adapted': adapted, 'baseline': baseline, 'trivial': trivial, 'verdict': 'sample-sanity',
              'reason': 'six held-out drawn scenes from the same generator; adaptation evidence, not a segmentation benchmark',
              'needs': 'panoptic annotations from the deployment domain (see the BYOD dataset branch) for any claim beyond the drawn vocabulary',
              'model_id': MODEL_ID, 'model_revision': MODEL_REVISION}
with open('outputs/mask2former_panoptic_finetune_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(evaluation, handle, indent=2, ensure_ascii=False, default=str)

## 9. Inference on new data

`shape_dataset` with a **different seed** produces scenes the model has never seen, in training or in the held-out split. Each is validated through `validate_inputs` (the inference contract: image type, sides, thresholds) and segmented with the adapted pipeline — the same `segment` call, the same post-processing, now over the five-class vocabulary. Their instance maps are known too, so the cell reports PQ on them as a second, independent sample (not used for any decision). The smoke run's first new scene came back as `sky`, `ground` and a `box`, with no void.

In [ ]:
NEW_DATA_SEED = 123  # @param {type:"integer"}

new_records = shape_dataset(4, seed=NEW_DATA_SEED)
new_rows = []
for record in new_records:
    manifest = validate_inputs(record['image'], names=[record['id']])
    prediction = adapter.segment(record['image'])
    new_rows.append({'id': record['id'], 'input': manifest['inputs'][0], 'segments': [(s['label'], round(s['score'], 3), round(s['area_fraction'], 3)) for s in prediction['segments']], 'void_fraction': round(prediction['void_fraction'], 3), 'reference_instances': record['instances']})
    print(new_rows[-1])
new_quality = adapter.evaluate(new_records)
print({'new_data_pq': round(new_quality['pq'], 4), 'n_images': new_quality['n_images']})
preview = Image.new('RGB', (1280, 240))
for index, record in enumerate(new_records):
    preview.paste(record['image'], (320 * index, 0))
preview

## 10. Export the artifact, then reload it as if from a cold start

`save_artifact` writes a self-describing directory: the Transformers `config.json` (which now carries the five-class vocabulary), `model.safetensors` (the full adapted weights, about 190 MB — the artifact type DIMER serving consumes, not a notebook-only surrogate; ART1), the image processor configuration, and `dimer-adapted-manifest.json` naming the format, the base model identity and revision, the vocabulary, the training record and the SHA-256 of every file (ART5). No training data, cache or credential is included (ART6).

`load_artifact` is the fresh-reload check: it rebuilds the pipeline from the directory alone, without reference to the `adapter` object still in memory, and refuses an artifact whose format, base identity or file digests do not match — before importing any model library. The cell then re-scores the held-out split with the reloaded model and asserts the PQ is identical and the maps agree pixel for pixel (VER1–VER5); a reload that quietly lost the adaptation would stop the notebook here. Finally it tampers with a copy of the manifest and confirms the copy is refused, because "it loads" is only evidence if something that should not load is also tried.

In [ ]:
import shutil
from pathlib import Path

ARTIFACT_DIR = Path('outputs') / 'mask2former-panoptic-adapted'
shutil.rmtree(ARTIFACT_DIR, ignore_errors=True)
descriptor = adapter.save_artifact(ARTIFACT_DIR, notes='standalone tutorial run')
print(json.dumps(descriptor, indent=2))

reloaded = Mask2FormerPanopticPipeline.load_artifact(ARTIFACT_DIR)
print({'source': reloaded.source, 'class_names': list(reloaded.class_names), 'stuff_names': list(reloaded.stuff_names), 'adapted': reloaded.adapted, 'device': reloaded.device})
reloaded_quality = reloaded.evaluate(held_out)
print({'reloaded_pq': round(reloaded_quality['pq'], 6), 'adapted_pq': round(adapted['pq'], 6)})
assert abs(reloaded_quality['pq'] - adapted['pq']) < 1e-9, 'the reloaded artifact does not reproduce the adapted score'
agreement = float((reloaded.segment(held_out[0]['image'])['segmentation'] == adapter.segment(held_out[0]['image'])['segmentation']).mean())
print({'pixel_agreement_first_held_out_scene': agreement})
assert agreement == 1.0, 'the reloaded artifact produces a different map'
print('fresh reload reproduces the adapted score and map exactly')

tampered = Path('outputs') / 'tampered-artifact'
shutil.rmtree(tampered, ignore_errors=True)
shutil.copytree(ARTIFACT_DIR, tampered)
with open(tampered / ARTIFACT_MANIFEST_NAME, encoding='utf-8') as handle:
    tampered_manifest = json.load(handle)
tampered_manifest['base']['revision'] = '0' * 40
with open(tampered / ARTIFACT_MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(tampered_manifest, handle)
try:
    Mask2FormerPanopticPipeline.load_artifact(tampered)
except ValueError as exc:
    print('tampered artifact refused ->', exc)
else:
    raise AssertionError('a tampered artifact was accepted')
finally:
    shutil.rmtree(tampered, ignore_errors=True)

## 11. Machine-readable outputs and provenance

Everything the notebook established, written to `outputs/mask2former_panoptic_finetune_result.json` beside the artifact: the pinned identity and manifest digests, the runtime, the COCO observation on the drawn scene, the dataset manifest and split, the training record with its loss history, the trivial / baseline / adapted / reloaded scores, the new-data rows, and the artifact descriptor. A later reader can tell what was measured, on what, with which weights, without rerunning anything. No credentials are recorded.

In [ ]:
payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'manifest_sha256': [entry['sha256'] for entry in MANIFEST['files']]},
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': adapter.device, 'cuda': torch.cuda.is_available(), 'precision': 'float32'},
    'coco_observation': coco_observation,
    'adaptation': {'dataset': dataset_manifest, 'split': evaluation['split'], 'run': run, 'trivial': trivial, 'baseline': baseline, 'adapted': adapted, 'reloaded': reloaded_quality, 'new_data': new_rows, 'new_data_quality': new_quality, 'artifact': descriptor},
}
with open('outputs/mask2former_panoptic_finetune_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir('outputs')))

## 12. Optional: your own image, or your own labelled dataset

Both branches are off by default so the sample path never opens an upload dialog. `USE_BYOD_IMAGE` uploads one image and segments it with the **adapted** model — remember it knows only the five drawn classes now. `USE_BYOD_DATASET` uploads a zip of a labelled directory (`dataset.json` + images + instance-id PNGs; the schema is printed) and runs it through the same stages as the sample: `validate_dataset`, `split_records`, a fresh re-headed pipeline for *your* vocabulary, `finetune`, `evaluate` against the trivial baseline, `save_artifact` and `load_artifact`. That is the adaptation contract (§25.10): a BYOD path that only ran inference would not be the workflow this notebook demonstrates. Ceilings are the ones `validate_dataset` enforces; a rejected dataset stops with the pipeline's own message.

In [ ]:
import io

USE_BYOD_IMAGE = False  # @param {type:"boolean"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_EPOCHS = 6  # @param {type:"integer"}
print('BYOD dataset schema:', json.dumps(DATASET_SCHEMA, indent=2))
if USE_BYOD_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    byod_name = next(iter(uploaded))
    byod_image = Image.open(io.BytesIO(uploaded[byod_name]))
    byod_image.load()
    print(validate_inputs(byod_image, names=[byod_name])['inputs'])
    byod_result = reloaded.segment(byod_image)
    print({'segments': [(s['label'], round(s['score'], 3), round(s['area_fraction'], 3)) for s in byod_result['segments']], 'void_fraction': round(byod_result['void_fraction'], 3)})
if USE_BYOD_DATASET:
    import zipfile

    from google.colab import files
    uploaded = files.upload()
    archive = next(iter(uploaded))
    target = Path('byod_dataset')
    shutil.rmtree(target, ignore_errors=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[archive])) as zf:
        for member in zf.infolist():
            destination = (target / member.filename).resolve()
            if not str(destination).startswith(str(target.resolve())):
                raise ValueError(f'refusing archive member outside the target directory: {member.filename}')
            zf.extract(member, target)
    root = target if (target / 'dataset.json').is_file() else next(p.parent for p in target.rglob('dataset.json'))
    byod_records, byod_classes, byod_stuff = load_labelled_dir(root)
    byod_manifest = validate_dataset(byod_records, byod_classes, byod_stuff, epochs=BYOD_EPOCHS)
    print(json.dumps({k: v for k, v in byod_manifest.items() if k != 'schema'}, indent=2))
    byod_train, byod_held = split_records(byod_records, holdout=HOLDOUT, seed=SEED)
    byod_adapter = Mask2FormerPanopticPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=byod_classes, stuff_names=byod_stuff, seed=SEED)
    byod_baseline = byod_adapter.evaluate(byod_held)
    byod_run = byod_adapter.finetune(byod_train, epochs=BYOD_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, freeze_backbone=FREEZE_BACKBONE, progress=lambda row: print(row))
    byod_adapted = byod_adapter.evaluate(byod_held)
    print({'baseline_pq': round(byod_baseline['pq'], 4), 'adapted_pq': round(byod_adapted['pq'], 4), 'per_class': {k: round(v['pq'], 3) for k, v in byod_adapted['per_class'].items()}})
    byod_dir = Path('outputs') / 'mask2former-panoptic-byod-adapted'
    shutil.rmtree(byod_dir, ignore_errors=True)
    print(byod_adapter.save_artifact(byod_dir, notes='BYOD dataset run'))
    byod_reloaded = Mask2FormerPanopticPipeline.load_artifact(byod_dir)
    assert abs(byod_reloaded.evaluate(byod_held)['pq'] - byod_adapted['pq']) < 1e-9
    print('BYOD artifact reloads and reproduces its held-out score')

## Interpretation and limits

The held-out panoptic quality is a **sample metric on drawn shapes**: six scenes from the same generator as the training scenes, three thing classes with distinct silhouettes, two stuff classes that always occupy the same halves of the image. Reaching PQ near 0.9 from 0.0 (the smoke run's 0.866) shows that the re-headed model, the upstream set loss, the bounded loop, the evaluation and the artifact boundary all work together; it does not show that 18 photographs of your own would train as easily, that the frozen backbone suits your domain, or that `box` (the weakest class at 0.509, confusable with the ground band's straight edges) would not need more data. **Training loss is not task evidence**, the class scores remain uncalibrated softmaxes, and the adapted model has forgotten COCO's vocabulary entirely — it knows five classes. There is no dispersion estimate: one seed, one split.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned base model, validate a labelled dataset, adapt the model with a bounded gradient fine-tune, evaluate it on held-out data, export a self-describing artifact and reload it from disk with identical outputs, in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** set `EPOCHS` to 2 and watch PQ fall; raise `N_IMAGES` to 60 and see whether `box` catches up; set `FREEZE_BACKBONE = False` and compare (the model card records why the default is frozen); change `DATASET_SEED` and see how much the held-out PQ moves between seeds — that spread is the dispersion this notebook does not otherwise report; then bring your own labelled directory through `USE_BYOD_DATASET`.

## References

- Repository README: https://github.com/kurtvalcorza/mask2former-panoptic-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/mask2former-panoptic-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/mask2former-panoptic-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/facebook/mask2former-swin-tiny-coco-panoptic
- Upstream code: https://github.com/facebookresearch/Mask2Former
- Masked-attention Mask Transformer for Universal Image Segmentation (Cheng et al., 2021): https://arxiv.org/abs/2112.01527
- Panoptic Segmentation (Kirillov et al., 2019): https://arxiv.org/abs/1801.00868